In [79]:
# import snowflake
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# ----- 1) Imports & Reproducibility -----
import os
import math
import random
import numpy as np
import pandas as pd
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

In [166]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat/landsat_features_training_mvdb.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]

def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

#ullify all negative observations
for column in wq_data.columns:
    if column != "Sample Date": wq_data[wq_data[column] < -9000][column] = np.nan 

wq_data['Month_cosine'] = np.cos((wq_data['Sample Date'].dt.month + (wq_data['Sample Date'].dt.day/31))* np.pi / 6)

#wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Sample Date',  'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#, 'Atmos opacity, 'Emsd', 'Lwir''
feature_cols = wq_data.columns.tolist()

#print(wq_data.columns)
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Month_cosine')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')

wq_data = wq_data.dropna(subset=feature_cols)

wq_data = wq_data.drop(columns='Sample Date')


In [167]:
print(wq_data.columns)

Index(['Latitude', 'Longitude', 'Total Alkalinity', 'Electrical Conductance',
       'Dissolved Reactive Phosphorus', 'Emis', 'Atran', 'Swir16',
       'Atmos_Opacity', 'Green', 'Drad', 'Red', 'Swir22', 'Emsd', 'Cdist',
       'Nir08', 'Blue', 'Trad', 'Lwir', 'Urad', 'Nvdi', 'Savi', 'Bsi', 'Ndbi',
       'Tcwi', 'Evi', 'Osavi', 'Gndvi', 'Gcvi', 'Msi', 'Bui', 'Tc Brightness',
       'Tc Greenness', 'Nbr', 'Ndsi', 'Green/Red Ratio', 'Ndgi',
       'Ui (Urban Index)', 'Nbr2', 'Red/Nir Ratio', 'Green/Nir Ratio', 'Aet',
       'Def', 'Pet', 'Ppt', 'Ppt_Roll3', 'Ppt_Roll6', 'Ppt_Roll12', 'Q',
       'Soil', 'Tmax', 'Tmin', 'Vpd', 'Et_Deficit', 'Pet_Ppt_Ratio',
       'Runoff_Coefficient', 'Temp_Range', 'Aet_Pet_Ration', 'Temp_Mean',
       'Month_cosine'],
      dtype='object')


In [169]:
wq_data_mini = wq_data[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil']]

In [193]:
Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat/landsat_features_validation_mvdb.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]

wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v)

#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)

#wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Atmos_Opacity', 'Emsd', 'Lwir', 'Unnamed: 0'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 

wq_data_v['Month_cosine'] = np.cos((wq_data_v['Sample Date'].dt.month + (wq_data_v['Sample Date'].dt.day/31))* np.pi / 6)

wq_data_final = wq_data_v[['Latitude', 'Longitude', 'Sample Date', 'Month_cosine']]


wq_data_v = wq_data_v.dropna(subset=feature_cols)

wq_data_v = wq_data_v.drop(columns='Sample Date')

wq_data_v = wq_data_v[sorted(wq_data_v.columns)]

print(wq_data_final.columns)

Index(['Latitude', 'Longitude', 'Sample Date', 'Month_cosine'], dtype='object')


In [194]:
wq_data_v_mini = wq_data_v[['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Nvdi', 'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil']]

In [195]:
wq_data_v.shape

(200, 60)

In [196]:
#number of cv groups
cv_groups = 8

#split over longitude
wq_data_mini['cv_group'] = pd.qcut(wq_data_mini['Latitude'], q=cv_groups, labels=False)

# Specify the number of folds
lat_sep_kf = GroupKFold(n_splits=cv_groups - 1)

In [197]:
#test train based on location
wq_data_test = wq_data_mini[wq_data_mini['cv_group'] == 7]
wq_data_train = wq_data_mini[wq_data_mini['cv_group'] != 7]

wq_data_test = wq_data_test.drop(columns='cv_group')
wq_data_train = wq_data_train.drop(columns='cv_group')

wq_data_train = wq_data_train[sorted(wq_data_train.columns)]
wq_data_test = wq_data_test[sorted(wq_data_test.columns)]

#then split into X and Y 
#Y_train = wq_data_train[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
#X_train = wq_data_train.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus", "Longitude", "Latitude"])
#Y_test = wq_data_test[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
#X_test = wq_data_test.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"])

In [198]:
null_counts = wq_data_train.isnull().sum()
print(null_counts)
null_counts = wq_data_test.isnull().sum()
print(null_counts)
null_counts = wq_data_v_mini.isnull().sum()
print(null_counts)

Aet                              0
Bsi                              0
Dissolved Reactive Phosphorus    0
Electrical Conductance           0
Latitude                         0
Longitude                        0
Month_cosine                     0
Ndbi                             0
Nvdi                             0
Pet                              0
Ppt                              0
Q                                0
Savi                             0
Soil                             0
Tmax                             0
Total Alkalinity                 0
Vpd                              0
dtype: int64
Aet                              0
Bsi                              0
Dissolved Reactive Phosphorus    0
Electrical Conductance           0
Latitude                         0
Longitude                        0
Month_cosine                     0
Ndbi                             0
Nvdi                             0
Pet                              0
Ppt                              0
Q      

In [199]:
# For reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)



Device: cpu


In [200]:
# ----- 2) Configurable column names & task type -----
# Adjust these lists for your real dataset
key_cols     = ["Latitude", "Longitude", "Month_cosine"]  # 3 key columns (IDs, timestamps, etc.)
feature_cols = wq_data_mini.columns.tolist()
feature_cols.remove('Latitude')
feature_cols.remove('Longitude')
feature_cols.remove('Month_cosine')
feature_cols.remove('Total Alkalinity')
feature_cols.remove('Electrical Conductance')
feature_cols.remove('Dissolved Reactive Phosphorus')
feature_cols.remove('cv_group')
#feature_cols = [f"f{i}" for i in range(1, 21)]  # 20 feature columns
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
#target_cols  = ["y1", "y2", "y3"]  # 3 target columns

# Choose the task: "regression" (default) or "multilabel"
#  - regression:  continuous y1,y2,y3  -> MSELoss
#  - multilabel:  binary y1,y2,y3 in {0,1} -> BCEWithLogitsLoss
task_type = "regression"


In [178]:
# # ----- 3) Create a synthetic dataset (replace this block with pd.read_csv for your data) -----
# N = 5000  # number of rows

# # Create keys (not used as features, but kept to reattach predictions later)
# df = pd.DataFrame({
#     "Latitude": np.arange(wq_data),
#     "Longitude": np.random.choice(list("ABCDE"), size=N),
# })

# # Create 20 numeric features
# for i in range(1, 21):
#     df[f"f{i}"] = np.random.randn(N) * (0.5 + i * 0.05) + i  # different scales

# # Create 3 targets:
# #   - For regression, continuous targets are noisy combos of features.
# #   - For multilabel (if you switch), we threshold them later.
# y1 = 0.8*df["f1"] - 0.3*df["f2"] + 0.5*df["f3"] + 0.1*np.random.randn(N)
# y2 = -0.2*df["f4"] + 1.1*df["f5"] + 0.1*df["f6"] + 0.1*np.random.randn(N)
# y3 = 0.4*df["f7"] + 0.2*df["f8"] - 0.5*df["f9"] + 0.1*np.random.randn(N)

# if task_type == "regression":
#     df["y1"], df["y2"], df["y3"] = y1, y2, y3
# else:  # multilabel example targets (0/1)
#     df["y1"] = (y1 > y1.mean()).astype(int)
#     df["y2"] = (y2 > y2.mean()).astype(int)
#     df["y3"] = (y3 > y3.mean()).astype(int)

In [179]:
# # ----- 4) Basic cleaning: keep only needed columns, handle NaNs -----
# needed_cols = key_cols + feature_cols + target_cols
# df = df[needed_cols].copy()
# df = df.fillna(0)


In [180]:
# # ----- 5) Train / Val / Test Split (70 / 15 / 15) -----
# indices = np.arange(len(df))
# np.random.shuffle(indices)

# n_train = int(0.70 * len(df))
# n_val   = int(0.15 * len(df))
# train_idx = indices[:n_train]
# val_idx   = indices[n_train:n_train + n_val]
# test_idx  = indices[n_train + n_val:]

# df_train = df.iloc[train_idx].reset_index(drop=True)
# df_val   = df.iloc[val_idx].reset_index(drop=True)
# df_test  = df.iloc[test_idx].reset_index(drop=True)


In [181]:
print(wq_data_train.shape)
print(wq_data_test.shape)
print(wq_data_v_mini.shape)

(8197, 17)
(1106, 17)
(200, 17)


In [182]:
print(wq_data_train.columns)
print(wq_data_test.columns)
print(wq_data_v_mini.columns)

Index(['Aet', 'Bsi', 'Dissolved Reactive Phosphorus', 'Electrical Conductance',
       'Latitude', 'Longitude', 'Month_cosine', 'Ndbi', 'Nvdi', 'Pet', 'Ppt',
       'Q', 'Savi', 'Soil', 'Tmax', 'Total Alkalinity', 'Vpd'],
      dtype='object')
Index(['Aet', 'Bsi', 'Dissolved Reactive Phosphorus', 'Electrical Conductance',
       'Latitude', 'Longitude', 'Month_cosine', 'Ndbi', 'Nvdi', 'Pet', 'Ppt',
       'Q', 'Savi', 'Soil', 'Tmax', 'Total Alkalinity', 'Vpd'],
      dtype='object')
Index(['Latitude', 'Longitude', 'Month_cosine', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Nvdi',
       'Savi', 'Ppt', 'Q', 'Bsi', 'Ndbi', 'Tmax', 'Vpd', 'Pet', 'Aet', 'Soil'],
      dtype='object')


In [183]:
wq_data_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1106 entries, 6 to 9298
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Aet                            1106 non-null   float64
 1   Bsi                            1106 non-null   float64
 2   Dissolved Reactive Phosphorus  1106 non-null   float64
 3   Electrical Conductance         1106 non-null   float64
 4   Latitude                       1106 non-null   float64
 5   Longitude                      1106 non-null   float64
 6   Month_cosine                   1106 non-null   float64
 7   Ndbi                           1106 non-null   float64
 8   Nvdi                           1106 non-null   float64
 9   Pet                            1106 non-null   float64
 10  Ppt                            1106 non-null   float64
 11  Q                              1106 non-null   float64
 12  Savi                           1106 non-null   float6

In [201]:
# ----- 6) Compute normalization stats on TRAIN ONLY -----
# Standardize features to zero-mean, unit-std
feat_means = wq_data_train[feature_cols].mean()
feat_stds  = wq_data_train[feature_cols].std().replace(0, 1.0)  # avoid division by zero

print(feat_means)
print(feat_stds)

def normalize_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out[feature_cols] = (out[feature_cols] - feat_means) / feat_stds
    return out

#scaler = StandardScaler()
#df_train_n = scaler.fit_transform(wq_data_train)
#df_val_n = scaler.transform(wq_data_test)
#df_test_n = scaler.transform(wq_data_v)



df_train_n = normalize_features(wq_data_train)
df_val_n   = normalize_features(wq_data_test)
df_test_n  = normalize_features(wq_data_v_mini)

# (Optional) You could also normalize targets for regression if scales differ greatly.
# For simplicity, we leave targets as-is.




Nvdi      0.148015
Savi      0.222018
Ppt      67.211651
Q         4.117946
Bsi       0.012258
Ndbi     -0.024011
Tmax     29.980458
Vpd       1.406704
Pet     176.219351
Aet      62.804868
Soil      8.193437
dtype: float64
Nvdi     0.096405
Savi     0.144604
Ppt     54.560163
Q        7.502649
Bsi      0.055027
Ndbi     0.079116
Tmax     2.926110
Vpd      0.510103
Pet     30.252998
Aet     45.165217
Soil    16.005774
dtype: float64


In [202]:
# print(df_train_n.shape)
# print(df_val_n.shape)
# print(df_test_n.shape)

In [ ]:
# df_test_n.head(5)

In [203]:
# ----- 7) PyTorch Dataset -----
class TabularDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, key_cols, feature_cols, target_cols, task_type="regression"):
        self.keys = frame[key_cols].reset_index(drop=True)
        X = frame[feature_cols].to_numpy(dtype=np.float32)
        self.X = torch.from_numpy(X)
        y = frame[target_cols].to_numpy(dtype=np.float32)
        self.y = torch.from_numpy(y)
        self.task_type = task_type

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # Return keys (as dict-like), features, and targets
        return self.keys.iloc[idx].to_dict(), self.X[idx], self.y[idx]

train_ds = TabularDataset(df_train_n, key_cols, feature_cols, target_cols, task_type=task_type)
val_ds   = TabularDataset(df_val_n,   key_cols, feature_cols, target_cols, task_type=task_type)
test_ds  = TabularDataset(df_test_n,  key_cols, feature_cols, target_cols, task_type=task_type)



In [204]:
# ----- 8) DataLoaders -----
BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [205]:
# ----- 9) Model: a simple MLP -----
class MLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, hidden_dims=(128, 64), dropout=0.1, task_type="regression"):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
        self.task_type = task_type

    def forward(self, x):
        logits_or_outputs = self.net(x)
        # For regression, return raw outputs.
        # For multilabel classification: during training we keep raw logits (BCEWithLogitsLoss expects logits).
        return logits_or_outputs

model = MLP(in_dim=len(feature_cols), out_dim=len(target_cols), hidden_dims=(128, 64), dropout=0.15, task_type=task_type).to(DEVICE)



In [206]:
# ----- 10) Loss and Optimizer -----
if task_type == "regression":
    criterion = nn.MSELoss()
else:  # "multilabel"
    criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [207]:
# ----- 11) Metrics -----
@torch.no_grad()
def evaluate(model, loader, task_type="regression", threshold=0.5):

    preds_list = []
    targets_list = []


    model.eval()
    total_loss = 0.0
    n_examples = 0

    # For regression: track MAE and RMSE
    mae_sum = torch.zeros(len(target_cols), device=DEVICE)
    mse_sum = torch.zeros(len(target_cols), device=DEVICE)

    # For multilabel: track accuracy (micro)
    correct = 0
    total   = 0

    for _, X, y in loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        out = model(X)

        preds_list.append(out.detach().cpu())
        targets_list.append(y.detach().cpu())

        loss = criterion(out, y)

        if torch.isnan(loss) or torch.isinf(loss):
            print("Bad loss detected!")
            print("out min/max:", out.min().item(), out.max().item())
            print("y min/max:", y.min().item(), y.max().item())
            print("loss:", loss)
            return {"loss": float("nan")}

        bs = y.size(0)
        total_loss += loss.item() * bs
        n_examples += bs

        if task_type == "regression":
            err = torch.abs(out - y)
            mae_sum += err.sum(dim=0)
            mse_sum += ((out - y) ** 2).sum(dim=0)
        else:
            # multilabel: apply sigmoid then threshold
            probs = torch.sigmoid(out)
            preds = (probs >= threshold).float()
            correct += (preds == y).sum().item()
            total   += y.numel()

    avg_loss = total_loss / max(n_examples, 1)

    if task_type == "regression":

        preds = torch.cat(preds_list, dim=0)
        targets = torch.cat(targets_list, dim=0)

        mae = (mae_sum / n_examples).detach().cpu().numpy()
        rmse = torch.sqrt(mse_sum / n_examples).detach().cpu().numpy()

        # Compute R2 per target
        ss_res = torch.sum((preds - targets) ** 2, dim=0)
        ss_tot = torch.sum((targets - targets.mean(dim=0)) ** 2, dim=0)

        r2 = 1 - ss_res / torch.clamp(ss_tot, min=1e-12)
        r2 = r2.numpy()

        metrics = {
            "loss": avg_loss,
            "MAE_per_target": {t: float(mae[i]) for i, t in enumerate(target_cols)},
            "RMSE_per_target": {t: float(rmse[i]) for i, t in enumerate(target_cols)},
            "R2_per_target": {t: float(r2[i]) for i, t in enumerate(target_cols)},
        }
    else:
        accuracy = correct / max(total, 1)
        metrics = {"loss": avg_loss, "micro_accuracy": accuracy}

    return metrics

In [208]:
# ----- 12) Training Loop -----
EPOCHS = 50
best_val = math.inf
patience = 10
wait = 0
best_state = None



for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_seen = 0

    for _, X, y in train_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        out = model(X)
        
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        n_seen += X.size(0)

    train_loss = running_loss / max(n_seen, 1)
    val_metrics = evaluate(model, val_loader, task_type=task_type)
    val_loss = val_metrics["loss"]

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Metrics: {val_metrics}")

    # Simple early stopping on validation loss
    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

# Load best model (if available)
if best_state is not None:
    model.load_state_dict(best_state)

Epoch 01 | Train Loss: 124464.6612 | Val Loss: 110064.3509 | Val Metrics: {'loss': 110064.35087025317, 'MAE_per_target': {'Total Alkalinity': 112.71115112304688, 'Electrical Conductance': 461.084228515625, 'Dissolved Reactive Phosphorus': 39.41142654418945}, 'RMSE_per_target': {'Total Alkalinity': 131.33059692382812, 'Electrical Conductance': 555.4219970703125, 'Dissolved Reactive Phosphorus': 66.72127532958984}, 'R2_per_target': {'Total Alkalinity': -2.7670538425445557, 'Electrical Conductance': -2.210521936416626, 'Dissolved Reactive Phosphorus': -0.5166953802108765}}
Epoch 02 | Train Loss: 104313.7384 | Val Loss: 79438.9888 | Val Metrics: {'loss': 79438.98881103074, 'MAE_per_target': {'Total Alkalinity': 69.9520263671875, 'Electrical Conductance': 378.60955810546875, 'Dissolved Reactive Phosphorus': 38.157493591308594}, 'RMSE_per_target': {'Total Alkalinity': 83.38485717773438, 'Electrical Conductance': 477.76788330078125, 'Dissolved Reactive Phosphorus': 55.693626403808594}, 'R2_pe

In [209]:
# ----- 13) Final evaluation on TEST -----
test_metrics = evaluate(model, val_loader, task_type=task_type)
print("TEST:", test_metrics)


TEST: {'loss': 34278.3414613472, 'MAE_per_target': {'Total Alkalinity': 56.18099594116211, 'Electrical Conductance': 244.23179626464844, 'Dissolved Reactive Phosphorus': 40.24973678588867}, 'RMSE_per_target': {'Total Alkalinity': 70.29798889160156, 'Electrical Conductance': 308.6703186035156, 'Dissolved Reactive Phosphorus': 51.1453857421875}, 'R2_per_target': {'Total Alkalinity': -0.07933390140533447, 'Electrical Conductance': 0.008439481258392334, 'Dissolved Reactive Phosphorus': 0.10878467559814453}}


In [131]:
# ----- 14) Inference: get predictions + keys side-by-side -----
@torch.no_grad()
def predict_with_keys(model, loader, task_type="regression", threshold=0.5):
    model.eval()
    all_rows = []
    for keys, X, _ in loader:
        X = X.to(DEVICE)
        logits_or_outputs = model(X).cpu().numpy()

        if task_type == "regression":
            preds = logits_or_outputs  # raw outputs
        else:
            probs = 1 / (1 + np.exp(-logits_or_outputs))  # sigmoid
            preds = (probs >= threshold).astype(np.float32)

        # For each batch row, combine keys + predictions into a dict
        for i in range(len(keys["Latitude"])):
            row = {k: keys[k][i] for k in keys.keys()}
            for j, t in enumerate(target_cols):
                row[f"pred_{t}"] = float(preds[i, j])
            all_rows.append(row)
    return pd.DataFrame(all_rows)

pred_df = predict_with_keys(model, test_loader, task_type=task_type)
pred_df.head(5)

,Latitude,Longitude,Month_cosine,pred_Total Alkalinity,pred_Electrical Conductance,pred_Dissolved Reactive Phosphorus
0,"tensor(-32.0433, dtype=torch.float64)","tensor(27.8228, dtype=torch.float64)","tensor(0.0169, dtype=torch.float64)",109.605881,460.890900,38.496590
1,"tensor(-33.3292, dtype=torch.float64)","tensor(26.0775, dtype=torch.float64)","tensor(0.2670, dtype=torch.float64)",91.329628,461.223145,33.757713
2,"tensor(-32.9916, dtype=torch.float64)","tensor(27.6400, dtype=torch.float64)","tensor(-0.9190, dtype=torch.float64)",95.916000,491.052490,35.924728
3,"tensor(-34.0964, dtype=torch.float64)","tensor(24.4392, dtype=torch.float64)","tensor(0.3944, dtype=torch.float64)",90.601540,463.063110,34.221146
4,"tensor(-32.0006, dtype=torch.float64)","tensor(28.5817, dtype=torch.float64)","tensor(0.5146, dtype=torch.float64)",115.922523,434.160156,38.969944


In [132]:
pred_df['Latitude'] = pred_df['Latitude'].apply(lambda x: x.item())
pred_df['Longitude'] = pred_df['Longitude'].apply(lambda x: x.item())
pred_df['Month_cosine'] = pred_df['Month_cosine'].apply(lambda x: x.item())

In [133]:
# Reverse cosine encoding
def reverse_month_cosine(cos_val):
    # Ensure value is in valid range for arccos
    cos_val = np.clip(cos_val, -1, 1)
    
    # Get angle in radians
    angle = np.arccos(cos_val)
    
    # Convert back to fractional month
    fractional_month = angle / (np.pi / 6)  # inverse of * np.pi/6
    
    # Approximate month and day
    month = int(fractional_month)  # integer month
    day = int(round((fractional_month - month) * 31))  # approximate day
    
    # Adjust for month starting at 1
    month = max(1, min(12, month))
    day = max(1, min(31, day))
    
    return month, day

# # Apply to DataFrame
# pred_df[['Approx_Month', 'Approx_Day']] = pred_df['Month_cosine'].apply(
#     lambda x: pd.Series(reverse_month_cosine(x))
# )

In [134]:
display(pred_df)

,Latitude,Longitude,Month_cosine,pred_Total Alkalinity,pred_Electrical Conductance,pred_Dissolved Reactive Phosphorus
0,-32.043333,27.822778,0.016889,109.605881,460.890900,38.496590
1,-33.329167,26.077500,0.266967,91.329628,461.223145,33.757713
2,-32.991639,27.640028,-0.918958,95.916000,491.052490,35.924728
3,-34.096389,24.439167,0.394356,90.601540,463.063110,34.221146
4,-32.000556,28.581667,0.514555,115.922523,434.160156,38.969944
...,...,...,...,...,...,...
195,-33.771111,25.386667,0.994869,101.084854,524.856445,37.763626
196,-33.185361,27.390750,0.067510,95.574615,491.575226,35.776726
197,-32.043333,27.822778,0.455495,110.377563,501.285278,39.506447
198,-33.001667,25.161389,0.790776,160.186432,661.331726,54.500664


In [ ]:
# # ----- 15) (Optional) Save artifacts -----
# os.makedirs("artifacts", exist_ok=True)
# torch.save({
#     "model_state_dict": model.state_dict(),
#     "feature_cols": feature_cols,
#     "target_cols": target_cols,
#     "feat_means": feat_means.to_dict(),
#     "feat_stds": feat_stds.to_dict(),
#     "task_type": task_type
# }, "artifacts/model.pt")

# pred_df.to_csv("artifacts/test_predictions_with_keys.csv", index=False)
# #print("Saved: artifacts/model.pt and artifacts/test_predictions_with_keys.csv")

In [136]:
condition = (pred_df['pred_Total Alkalinity'] > 20) & (pred_df['pred_Total Alkalinity'] < 200)

count1 = condition.sum()

print("Count using sum():", count1)


condition = (pred_df['pred_Electrical Conductance'] < 800)

count1 = condition.sum()

print("Count using sum():", count1)

condition = (pred_df['pred_Dissolved Reactive Phosphorus'] < 100)

count1 = condition.sum()

print("Count using sum():", count1)


Count using sum(): 200
Count using sum(): 200
Count using sum(): 200


In [135]:
pred_df.to_csv('data/output.csv', index=False)

In [ ]:
# session.sql("""
#     PUT file://output.csv
#     'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge-version2"/versions/live/data/'
#     AUTO_COMPRESS=FALSE
#     OVERWRITE=TRUE
# """).collect()

# print("File saved! Refresh the browser to see the files in the sidebar")